In [4]:
import data.breathe_data as bd
import data.helpers as dh
from plotly.subplots import make_subplots
import pandas as pd
import plotly.graph_objs as go
import cfr.corr as corr

In [ ]:
# df1 = bd.load_meas_from_excel(
#     # "infer_all_19_data_with_best_FEV1",
#     "pppfev1_ppfev1st_ppfev1ft_IV_19_plus1820_assoc",
#     study_folder="CFR",
#     str_cols_to_arrays=[
#         # "Airway resistance (%)",
#         # "P(HFEV1|FEF2575, bFEV1, FEV1)",
#         "P(HFEV1|bFEV1)",
#         "P(HFEV1|FEV1)",
#     ],
#     use_csv=True,
#     bypass_sanity_checks=True,
# )

In [2]:
df = bd.load_meas_from_excel(
    # "infer_all_19_data_with_best_FEV1",
    "ppfev1st_ft_bFEV1_2016-19_IV_2019-21_assoc",
    study_folder="CFR",
    str_cols_to_arrays=[
        # "Airway resistance (%)",
        # "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|bFEV1)",
        "P(HFEV1|FEV1)",
    ],
    use_csv=True,
    bypass_sanity_checks=True,
)

# Viz ranked associations with IV days 2

In [24]:
diff_col = "ppFEV1FT - ppFEV1ST"
prctile = 50
# for prctile in [50, 60, 70, 80, 85, 90, 95]:
for prctile in [0, 10, 20, 30, 40]:
# for prctile in [50]:
    title = f"main viz - (|{diff_col}| > {prctile}th pctile)"

    df_ranked2 = df.copy()
    # df_ranked2 = df_ranked2[(df_ranked2["ecFEF2575%ecFEV1"] < 70)].copy()
    threshold = df_ranked2[diff_col].abs().quantile(prctile / 100)
    df_ranked2 = df_ranked2[df_ranked2[diff_col].abs() > threshold]

    rank_metric = "FEV1 % Predicted"
    df_ranked2.loc[df_ranked2[rank_metric] >= 70, "severity"] = "mild"
    df_ranked2.loc[
        (df_ranked2[rank_metric] >= 40) & (df_ranked2[rank_metric] < 70), "severity"
    ] = "moderate"
    df_ranked2.loc[df_ranked2[rank_metric] < 40, "severity"] = "severe"

    colors = {"FEV1%PredFT": "red", "FEV1%PredST": "blue"}
    # colors = {"FEV1%PredFT": "black", "FEV1%PredST": "grey"}
    legend_names = {"FEV1%PredFT": "Predicted", "FEV1%PredST": "Baseline"}

    # Shared IV days y-range per severity
    iv_range_by_severity = {}
    for sev in ["mild", "moderate"]:
        df_sev = df_ranked2[df_ranked2["severity"] == sev]["IV days"].dropna()
        iv_range_by_severity[sev] = [-5, df_sev.max() * 1.05]

    # 4 rows, 1 col: predicted mild, baseline mild, predicted moderate, baseline moderate
    panels = [
        ("mild", "FEV1%PredFT"),
        ("mild", "FEV1%PredST"),
        ("moderate", "FEV1%PredFT"),
        ("moderate", "FEV1%PredST"),
    ]

    fig = make_subplots(
        rows=4,
        cols=1,
        vertical_spacing=0.05,
        row_titles=["Mild CF", "", "Moderate CF", ""],
    )

    legend_shown = set()

    # Overall corr
    res_overall = corr.bootstrap_corr_diff(
        df_ranked2["FEV1%PredFT"],
        df_ranked2["FEV1%PredST"],
        df_ranked2["IV days"],
        n_bootstrap=1000,
    )
    ci_overall = f"[{res_overall['ci_lo_95']:.3f}; {res_overall['ci_hi_95']:.3f}]"
    corrs = f"<br>Corr diff: overall {ci_overall}"

    for row_idx, (severity_label, fev_metric) in enumerate(panels, start=1):
        df_sev = df_ranked2[df_ranked2["severity"] == severity_label].copy()
        df_sorted = df_sev.sort_values(fev_metric, ascending=False).reset_index(
            drop=True
        )
        x_rank = list(range(len(df_sorted)))

        # Per severity correlation
        if row_idx in [1, 3]:
            res_sev = corr.bootstrap_corr_diff(
                df_sev["FEV1%PredFT"],
                df_sev["FEV1%PredST"],
                df_sev["IV days"],
                n_bootstrap=1000,
                )
            ci_sev = f"[{res_sev['ci_lo_95']:.3f}; {res_sev['ci_hi_95']:.3f}]"
            corrs += f"<br>              {severity_label} {ci_sev}"

        fig.add_trace(
            go.Scatter(
                x=x_rank,
                y=df_sorted["IV days"],
                mode="markers",
                name=legend_names[fev_metric],
                marker=dict(size=3, color=colors[fev_metric]),
                customdata=df_sorted["ID"],
                hovertemplate="ID: %{customdata}<br>IV days: %{y:.0f}<extra></extra>",
                showlegend=False,
            ),
            row=row_idx,
            col=1,
        )
        fig.update_xaxes(
            showticklabels=False,
            title_text=f"Individuals (ranked by {fev_metric})",
            linecolor="black",
            linewidth=1,
            showline=True,
            row=row_idx,
            col=1,
        )
        fig.update_yaxes(
            title_text="IV days",
            range=iv_range_by_severity[severity_label],
            linecolor="black",
            linewidth=1,
            showline=True,
            row=row_idx,
            col=1,
        )

    fig.update_layout(
        height=800,
        width=700,
        title=title + corrs,
        margin=dict(t=150),
        template="simple_white",
        plot_bgcolor="white",
        paper_bgcolor="white",
    )

    # fig.show()
    fig.write_image(
        dh.get_path_to_main() + f"PlotsCFR/Ranked IV days and FEV1/{title}.pdf"
    )

In [ ]:
import numpy as np

N_BINS = 10
diff_col = "ppFEV1FT - ppFEV1ST"
rank_metric = "FEV1 % Predicted"
colors = {"FEV1%PredFT": "red", "FEV1%PredST": "blue"}
legend_names = {"FEV1%PredFT": "Predicted", "FEV1%PredST": "Baseline"}
severities = ["mild", "moderate"]
fev_metrics = ["FEV1%PredFT", "FEV1%PredST"]

for prctile in [50, 60, 70, 80, 85, 90]:
    df_ranked2 = df.copy()
    threshold = df_ranked2[diff_col].abs().quantile(prctile / 100)
    df_ranked2 = df_ranked2[df_ranked2[diff_col].abs() > threshold]

    df_ranked2.loc[df_ranked2[rank_metric] >= 70, "severity"] = "mild"
    df_ranked2.loc[
        (df_ranked2[rank_metric] >= 40) & (df_ranked2[rank_metric] < 70), "severity"
    ] = "moderate"
    df_ranked2.loc[df_ranked2[rank_metric] < 40, "severity"] = "severe"

    # Shared IV days y-range per severity (mean + std ceiling across both metrics)
    iv_range_by_severity = {}
    for sev in severities:
        df_sev = df_ranked2[df_ranked2["severity"] == sev]
        ceil = 0
        for fev_metric in fev_metrics:
            df_sorted = df_sev.sort_values(fev_metric, ascending=False).reset_index(
                drop=True
            )
            df_sorted["bin"] = pd.cut(df_sorted.index, bins=N_BINS, labels=False)
            grouped = df_sorted.groupby("bin", observed=False)["IV days"].agg(
                ["mean", "std"]
            )
            ceil = max(ceil, (grouped["mean"] + grouped["std"].fillna(0)).max() * 1.1)
        iv_range_by_severity[sev] = [0, ceil]

    bin_labels = [f"{i*10}–{(i+1)*10}%" for i in range(N_BINS)]

    fig = make_subplots(
        rows=2,
        cols=1,
        vertical_spacing=0.1,
        row_titles=["Mild CF", "Moderate CF"],
    )

    legend_shown = set()

    for row_idx, severity_label in enumerate(severities, start=1):
        df_sev = df_ranked2[df_ranked2["severity"] == severity_label].copy()

        for fev_metric in fev_metrics:
            df_sorted = df_sev.sort_values(fev_metric, ascending=False).reset_index(
                drop=True
            )
            df_sorted["bin"] = pd.cut(df_sorted.index, bins=N_BINS, labels=False)
            grouped = (
                df_sorted.groupby("bin", observed=False)["IV days"]
                .agg(["mean", "std", "count"])
                .reset_index()
            )

            show_legend = fev_metric not in legend_shown
            legend_shown.add(fev_metric)

            fig.add_trace(
                go.Bar(
                    x=bin_labels,
                    y=grouped["mean"],
                    error_y=dict(
                        type="data",
                        array=grouped["std"].fillna(0).tolist(),
                        visible=True,
                    ),
                    marker_color=colors[fev_metric],
                    name=legend_names[fev_metric],
                    customdata=grouped["count"],
                    hovertemplate="Bin: %{x}<br>Mean IV days: %{y:.1f}<br>n=%{customdata}<extra></extra>",
                    showlegend=show_legend,
                    opacity=0.7,
                ),
                row=row_idx,
                col=1,
            )

        fig.update_xaxes(
            linecolor="black", linewidth=1, showline=True, row=row_idx, col=1
        )
        fig.update_yaxes(
            title_text="IV days (mean ± SD)",
            range=iv_range_by_severity[severity_label],
            linecolor="black",
            linewidth=1,
            showline=True,
            row=row_idx,
            col=1,
        )

    fig.update_xaxes(title_text="Rank percentile (high → low FEV1)", row=2, col=1)

    title = f"Hist - IV days by FEV1 rank decile<br>(|{diff_col}| > {prctile}th pctile)"
    fig.update_layout(
        height=600,
        width=700,
        title=title,
        template="simple_white",
        plot_bgcolor="white",
        paper_bgcolor="white",
        legend=dict(x=1.02, y=1, xanchor="left"),
        barmode="overlay",
        bargap=0.1,
    )
    # fig.show()
    fig.write_image(
        dh.get_path_to_main() + f"PlotsCFR/Ranked FEV1 and IV days/{title}.pdf"
    )

# Viz ranked associations with IV days

In [ ]:
diff_col = "ppFEV1FT - ppFEV1ST"
prctile = 50  # keep rows where abs(diff_col) > this percentile threshold

df_ranked = df.copy()
df_ranked = df_ranked[(df_ranked["ecFEF2575%ecFEV1"] < 70)].copy()
threshold = df_ranked[diff_col].abs().quantile(prctile / 100)
df_ranked = df_ranked[df_ranked[diff_col].abs() > threshold]

rank_metric = "FEV1 % Predicted"

df_ranked.loc[df_ranked[rank_metric] >= 70, "severity"] = "mild"
df_ranked.loc[
    (df_ranked[rank_metric] >= 40) & (df_ranked[rank_metric] < 70), "severity"
] = "moderate"
df_ranked.loc[df_ranked[rank_metric] < 40, "severity"] = "severe"

iv_max = df_ranked["IV days"].max() * 1.05

# severities = [("mild", 1, 2), ("moderate", 3, 4), ("severe", 5, 6)]
severities = [("mild", 1, 2)]
fev_metrics = ["FEV1%PredST", "FEV1%PredFT"]
fev_colors = {"FEV1%PredST": "blue", "FEV1%PredFT": "red"}
iv_color = "rgba(64, 64, 64, 0.8)"

# Compute shared y range per severity across both fev_metrics
fev_range_by_severity = {}
for severity_label, _, _ in severities:
    df_sev = df_ranked[df_ranked["severity"] == severity_label]
    vals = pd.concat([df_sev[m] for m in fev_metrics]).dropna()
    pad = (vals.max() - vals.min()) * 0.05
    fev_range_by_severity[severity_label] = [vals.min() - pad, vals.max() + pad]


def add_ranked_fev_iv_panels(
    fig,
    df_severity,
    fev_metric,
    severity_label,
    row_fev,
    row_iv,
    col,
    overlay_metric=None,
):
    df_sorted = df_severity.sort_values(fev_metric, ascending=False).reset_index(
        drop=True
    )
    x_rank = list(range(len(df_sorted)))

    if overlay_metric is not None:
        fig.add_trace(
            go.Scatter(
                x=x_rank,
                y=df_sorted[overlay_metric],
                mode="markers",
                marker=dict(size=3, opacity=0.3, color=fev_colors[overlay_metric]),
                customdata=df_sorted["ID"],
                hovertemplate="ID: %{customdata}<br>"
                + overlay_metric
                + ": %{y:.1f}<extra></extra>",
                showlegend=False,
            ),
            row=row_fev,
            col=col,
        )

    fig.add_trace(
        go.Scatter(
            x=x_rank,
            y=df_sorted[fev_metric],
            mode="markers",
            marker=dict(size=3, opacity=1.0, color=fev_colors[fev_metric]),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>"
            + fev_metric
            + ": %{y:.1f}<extra></extra>",
            showlegend=False,
        ),
        row=row_fev,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=x_rank,
            y=df_sorted["IV days"],
            mode="markers",
            marker=dict(size=3, color=iv_color),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>IV days: %{y:.0f}<extra></extra>",
            showlegend=False,
        ),
        row=row_iv,
        col=col,
    )
    iv_max = df_sorted["IV days"].max() * 1.05
    fig.update_xaxes(
        showticklabels=False,
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_fev,
        col=col,
    )
    fig.update_xaxes(
        showticklabels=False,
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_iv,
        col=col,
    )
    fig.update_yaxes(
        title_text=f"{fev_metric}",
        linecolor="black",
        linewidth=1,
        showline=True,
        range=fev_range_by_severity[severity_label],
        row=row_fev,
        col=col,
    )
    fig.update_yaxes(
        title_text="IV days",
        range=[-5, iv_max],
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_iv,
        col=col,
    )


fig = make_subplots(
    rows=2,
    cols=2,
    vertical_spacing=0.03,
    horizontal_spacing=0.12,
    column_titles=["Baseline", "Predicted"],
    row_titles=[
        "Mild CF",
        "",
        "Moderate CF",
        "",
        "Severe CF",
        "",
    ],
)

for col_idx, fev_metric in enumerate(fev_metrics, start=1):
    overlay = "FEV1%PredST" if col_idx == 2 else None
    for severity_label, row_iv, row_fev in severities:
        df_sev = df_ranked[df_ranked["severity"] == severity_label]
        add_ranked_fev_iv_panels(
            fig, df_sev, fev_metric, severity_label, row_fev, row_iv, col_idx
        )
        add_ranked_fev_iv_panels(
            fig,
            df_sev,
            fev_metric,
            severity_label,
            row_fev,
            row_iv,
            col_idx,
            overlay_metric=overlay,
        )

title = f"Ranked FEV1 and IV days by severity (|{diff_col}| > {prctile}th pctile) 2"
fig.update_layout(
    height=700,
    width=1100,
    # height=1100, width=1100,
    title=title,
    template="simple_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Ranked FEV1 and IV days/{title}.pdf")

In [ ]:
# Archive
diff_col = "ppFEV1FT - ppFEV1ST"
prctile = 0  # keep rows where abs(diff_col) > this percentile threshold

df_ranked = df.copy()
# df_ranked = df_ranked[(df_ranked["ecFEF2575%ecFEV1"] < 70)].copy()
threshold = df_ranked[diff_col].abs().quantile(prctile / 100)
df_ranked = df_ranked[df_ranked[diff_col].abs() > threshold]

df_ranked.loc[df_ranked["FEV1%PredST"] >= 70, "severity"] = "mild"
df_ranked.loc[
    (df_ranked["FEV1%PredST"] >= 40) & (df_ranked["FEV1%PredST"] < 70), "severity"
] = "moderate"
df_ranked.loc[df_ranked["FEV1%PredST"] < 40, "severity"] = "severe"

iv_max = df_ranked["IV days"].max() * 1.05

severities = [("mild", 1, 2), ("moderate", 3, 4), ("severe", 5, 6)]
fev_metrics = ["FEV1%PredST", "FEV1%PredFT"]
fev_colors = {"FEV1%PredST": "blue", "FEV1%PredFT": "red"}
iv_color = "rgba(64, 64, 64, 0.8)"

# Compute shared y range per severity across both fev_metrics
fev_range_by_severity = {}
for severity_label, _, _ in severities:
    df_sev = df_ranked[df_ranked["severity"] == severity_label]
    vals = pd.concat([df_sev[m] for m in fev_metrics]).dropna()
    pad = (vals.max() - vals.min()) * 0.05
    fev_range_by_severity[severity_label] = [vals.min() - pad, vals.max() + pad]


def add_ranked_fev_iv_panels(
    fig,
    df_severity,
    fev_metric,
    severity_label,
    row_fev,
    row_iv,
    col,
    overlay_metric=None,
):
    df_sorted = df_severity.sort_values(fev_metric, ascending=False).reset_index(
        drop=True
    )
    x_rank = list(range(len(df_sorted)))

    if overlay_metric is not None:
        fig.add_trace(
            go.Scatter(
                x=x_rank,
                y=df_sorted[overlay_metric],
                mode="markers",
                marker=dict(size=3, opacity=0.3, color=fev_colors[overlay_metric]),
                customdata=df_sorted["ID"],
                hovertemplate="ID: %{customdata}<br>"
                + overlay_metric
                + ": %{y:.1f}<extra></extra>",
                showlegend=False,
            ),
            row=row_fev,
            col=col,
        )

    fig.add_trace(
        go.Scatter(
            x=x_rank,
            y=df_sorted[fev_metric],
            mode="markers",
            marker=dict(size=3, opacity=1.0, color=fev_colors[fev_metric]),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>"
            + fev_metric
            + ": %{y:.1f}<extra></extra>",
            showlegend=False,
        ),
        row=row_fev,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=x_rank,
            y=df_sorted["IV days"],
            mode="markers",
            marker=dict(size=3, color=iv_color),
            customdata=df_sorted["ID"],
            hovertemplate="ID: %{customdata}<br>IV days: %{y:.0f}<extra></extra>",
            showlegend=False,
        ),
        row=row_iv,
        col=col,
    )
    fig.update_xaxes(
        showticklabels=False,
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_fev,
        col=col,
    )
    fig.update_xaxes(
        showticklabels=False,
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_iv,
        col=col,
    )
    fig.update_yaxes(
        title_text=f"{fev_metric}",
        linecolor="black",
        linewidth=1,
        showline=True,
        range=fev_range_by_severity[severity_label],
        row=row_fev,
        col=col,
    )
    fig.update_yaxes(
        title_text="IV days",
        range=[-5, iv_max],
        linecolor="black",
        linewidth=1,
        showline=True,
        row=row_iv,
        col=col,
    )


fig = make_subplots(
    rows=6,
    cols=2,
    vertical_spacing=0.03,
    horizontal_spacing=0.12,
    column_titles=["Baseline", "Predicted"],
    row_titles=[
        "Mild CF",
        "",
        "Moderate CF",
        "",
        "Severe CF",
        "",
    ],
)

for col_idx, fev_metric in enumerate(fev_metrics, start=1):
    overlay = "FEV1%PredST" if col_idx == 2 else None
    for severity_label, row_fev, row_iv in severities:
        df_sev = df_ranked[df_ranked["severity"] == severity_label]
        add_ranked_fev_iv_panels(
            fig, df_sev, fev_metric, severity_label, row_fev, row_iv, col_idx
        )
        add_ranked_fev_iv_panels(
            fig,
            df_sev,
            fev_metric,
            severity_label,
            row_fev,
            row_iv,
            col_idx,
            overlay_metric=overlay,
        )

title = f"Ranked FEV1 and IV days by severity (|{diff_col}| > {prctile}th pctile) 2"
fig.update_layout(
    height=1100,
    width=900,
    title=title,
    template="simple_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
)
# fig.write_image(dh.get_path_to_main() + f"PlotsCFR/{title}.pdf")

## Evaluate reclassified individuals

In [19]:
df_mild = df_ranked[df_ranked["severity"] == "mild"]
df_reclassified = df_mild[df_ranked["FEV1%PredFT"] < 70]

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_13480/2390661838.py:2: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [ ]:
print(f"{df_reclassified.shape[0]} of mild patients reclassified as moderate")
print(
    f"{df_reclassified.shape[0]/df_mild.shape[0] * 100:.1f}% of mild patients reclassified as moderate"
)
df_reclassified["FEV1%PredFT"].describe()

74 of mild patients reclassified as moderate
9.1% of mild patients reclassified as moderate


count    74.000000
mean     67.148786
std       3.250502
min      55.578116
25%      66.642265
50%      68.216090
75%      69.345391
max      69.978072
Name: FEV1%PredFT, dtype: float64

In [22]:
df_reclassified.sort_values(by="FEV1%PredFT").head(5)

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,...,bFEV1 % diff,idx best FEV1 2016-19,P(HFEV1|FEV1),P(HFEV1|bFEV1),FEV1%PredST,FEV1%PredFT,ppFEV1FT - ppFEV1ST,IVs,IV days,severity
66,B157445,18,173,3.22,2.26,3.61,Male,2019-01-01,3.22,2.26,...,58.448758,114,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.266...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",75.885290,55.578116,-20.307174,0.000000,0.000000,mild
34,B157861,50,170,2.84,2.30,2.84,Female,2019-01-01,2.84,2.30,...,76.760569,100,"[0.0, 0.0, 1.23407231e-306, 1.07774138e-288, 2...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",88.558334,56.391122,-32.167211,0.000000,0.000000,mild
52,B164053,34,166,2.79,1.55,2.92,Female,2019-01-01,2.79,1.55,...,61.986297,94,"[0.0, 8.60667128e-312, 1.30789243e-293, 5.5568...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",83.881837,58.466162,-25.415676,0.333333,5.000000,mild
192,B167687,26,167,2.74,1.43,3.39,Female,2019-01-01,2.74,1.43,...,32.743359,90,"[6.19355526e-316, 1.51329715e-297, 1.02024786e...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",79.490345,59.359564,-20.130781,5.666667,90.000000,mild
188,B167849,29,158,2.25,1.23,2.73,Female,2019-01-01,2.25,1.23,...,33.333332,72,"[5.16854868e-190, 3.8968373e-175, 7.00232658e-...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",74.789247,60.035783,-14.753465,2.000000,21.333333,mild


# Correlations computation

In [34]:
import cfr.corr as corr

In [ ]:
# --- Configuration ---
baseline_col = "FEV1%PredST"  # baseline FEV metric; can also be "FEV1%Pred"
predicted_col = "FEV1%PredFT"  # always FEV1%PredFT
iv_col = "IV days"
diff_col = "ppFEV1FT - ppFEV1ST"
percentile_thresholds = [0, 25, 50, 75, 90]
N_BOOTSTRAP = 10_000

# Severity group definitions (based on baseline FEV1%PredST)
severity_groups = [
    ("Severe (<40)", df["FEV1%PredST"] < 40),
    ("Moderate (40-69)", (df["FEV1%PredST"] >= 40) & (df["FEV1%PredST"] < 70)),
    # ("Moderate 2 (40-49)", (df["FEV1%PredST"] >= 40) & (df["FEV1%PredST"] < 50)),
    # ("Moderate 1 (50-69)", (df["FEV1%PredST"] >= 50) & (df["FEV1%PredST"] < 70)),
    # ("Mild 3 (70-79)",     (df["FEV1%PredST"] >= 70) & (df["FEV1%PredST"] < 80)),
    # ("Mild 2 (80-89)",     (df["FEV1%PredST"] >= 80) & (df["FEV1%PredST"] < 90)),
    # ("Mild 1 (>=90)",       df["FEV1%PredST"] >= 90),
    ("Mild 1 (>=90)", df["FEV1%PredST"] >= 70),
]


def _analyze_subset(df_sub, label=""):
    n = len(df_sub)
    if n < 5:
        print(f"  [{label}] n={n}: skipping (too few samples)")
        return None
    res = corr.bootstrap_corr_diff(
        df_sub[baseline_col],
        df_sub[predicted_col],
        df_sub[iv_col],
        n_bootstrap=N_BOOTSTRAP,
    )
    sig_95 = "SIG**" if res["significant_95"] else "ns"
    sig_90 = "SIG*" if res["significant_90"] else "ns"
    print(
        f"  [{label}] n={res['n']:3d} | "
        f"r_base={res['r_baseline']:+.3f}(p={res['p_baseline']:.2e}), "
        f"r_pred={res['r_predicted']:+.3f}(p={res['p_predicted']:.2e}) | "
        f"diff={res['diff']:+.3f} | "
        f"95%CI=[{res['ci_lo_95']:+.3f},{res['ci_hi_95']:+.3f}] {sig_95} | "
        f"90%CI=[{res['ci_lo_90']:+.3f},{res['ci_hi_90']:+.3f}] {sig_90}"
    )
    return res


# --- Main Analysis ---
# all_results[prctile][label] = result dict
all_results = {}

for prctile in percentile_thresholds:
    print(f"\n{'='*100}")
    print(f"PERCENTILE THRESHOLD OF {diff_col}: {prctile}%")
    print(f"{'='*100}")
    all_results[prctile] = {}

    # Population level: threshold applied globally across all patients
    t_pop = df[diff_col].abs().quantile(prctile / 100)
    df_pop = df[df[diff_col].abs() >= t_pop].copy()
    print(f"\n  Population: abs({diff_col}) >= {t_pop:.2f}, n={len(df_pop)}")
    all_results[prctile]["Population"] = _analyze_subset(df_pop, label="Population")

    # By severity group: threshold applied within each group independently
    for group_name, group_mask in severity_groups:
        df_grp = df[group_mask].copy()
        if df_grp.empty:
            continue
        t_grp = df_grp[diff_col].abs().quantile(prctile / 100)
        df_sub = df_grp[df_grp[diff_col].abs() >= t_grp].copy()
        all_results[prctile][group_name] = _analyze_subset(df_sub, label=group_name)


PERCENTILE THRESHOLD OF ppFEV1FT - ppFEV1ST: 0%

  Population: abs(ppFEV1FT - ppFEV1ST) >= 0.00, n=2037
  [Population] n=2037 | r_base=-0.514(p=6.09e-138), r_pred=-0.523(p=5.08e-144) | diff=+0.010 | 95%CI=[+0.004,+0.015] SIG** | 90%CI=[+0.005,+0.014] SIG*
  [Severe (<40)] n=341 | r_base=-0.204(p=7.60e-05), r_pred=-0.203(p=7.78e-05) | diff=-0.000 | 95%CI=[-0.009,+0.007] ns | 90%CI=[-0.008,+0.006] ns
  [Moderate (40-69)] n=772 | r_base=-0.334(p=7.33e-22), r_pred=-0.353(p=2.12e-24) | diff=+0.019 | 95%CI=[+0.006,+0.034] SIG** | 90%CI=[+0.008,+0.031] SIG*
  [Mild 1 (>=90)] n=924 | r_base=-0.177(p=3.17e-08), r_pred=-0.219(p=8.72e-12) | diff=+0.042 | 95%CI=[+0.017,+0.067] SIG** | 90%CI=[+0.021,+0.063] SIG*

PERCENTILE THRESHOLD OF ppFEV1FT - ppFEV1ST: 25%

  Population: abs(ppFEV1FT - ppFEV1ST) >= 0.01, n=1528
  [Population] n=1528 | r_base=-0.454(p=5.60e-79), r_pred=-0.470(p=6.30e-85) | diff=+0.015 | 95%CI=[+0.007,+0.024] SIG** | 90%CI=[+0.008,+0.023] SIG*
  [Severe (<40)] n=256 | r_base=-0

# Viz baseline-prediction diff vs IV days

In [ ]:
diff_col = "ppFEV1FT - ppFEV1ST"

dftmp = df.copy()

# Filter percentile of certain population
prctile = 90
for prctile in [50, 60, 70, 80, 90]:
    t = dftmp[diff_col].abs().quantile(prctile / 100)
    print(f"{prctile}th percentile of absolute difference: {t}")

    dftmp = dftmp[dftmp[diff_col].abs() > t]

    # Filter by severity level
    dftmp.loc[dftmp["FEV1%PredST"] >= 70, "severity"] = "mild"
    dftmp.loc[
        (dftmp["FEV1%PredST"] >= 40) & (dftmp["FEV1%PredST"] < 70), "severity"
    ] = "moderate"
    dftmp.loc[dftmp["FEV1%PredST"] < 40, "severity"] = "severe"

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True)

    for i, severity in enumerate(["mild", "moderate", "severe"], start=1):
        dftmp_severity = dftmp[dftmp["severity"] == severity]
        fig.add_trace(
            go.Scatter(
                x=dftmp_severity[diff_col],
                y=dftmp_severity["IV days"],
                mode="markers",
                marker=dict(size=3, opacity=0.8),
            ),
            row=i,
            col=1,
        )
        fig.update_yaxes(
            title="IV days", range=[-5, dftmp["IV days"].max() * 1.1], row=i, col=1
        )
    fig.update_xaxes(title=f"{diff_col}", row=3, col=1)

    title = f"bFEV1_2016_19_IVdays_2019-21_{prctile}th_prctile"
    fig.update_layout(
        height=600,
        width=800,
        title=title,
    )

    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

    ##########################################################################################
    def labels_from_bins(bins):
        labels = [f"< {bins[1]}"]
        for i in range(1, len(bins) - 2):
            labels.append(f"[{bins[i]}, {bins[i+1]})")
        labels.append(f">= {bins[-2]}")
        return labels

    fig = make_subplots(3, 1)

    row = 0
    for severity in ["mild", "moderate", "severe"]:
        row += 1
        dftmp2 = dftmp[dftmp["severity"] == severity].copy()
        bins = [-1000, -20, -11, -9, -7, -5, -3, -1, 0]
        labels = labels_from_bins(bins)
        dftmp2["diff_bin"] = pd.cut(dftmp2[diff_col], bins=bins, labels=labels)

        grouped = (
            dftmp2.groupby("diff_bin", observed=False)
            .agg(
                iv_mean=("IV days", "mean"),
                iv_std=("IV days", "std"),
                diff_mean=(diff_col, "mean"),
                count=(diff_col, "size"),
            )
            .reset_index()
        )

        fig.add_trace(
            go.Bar(
                x=grouped["diff_bin"].astype(str),
                y=grouped["iv_mean"],
                # mode="markers+text",
                error_y=dict(
                    type="data", array=grouped["iv_std"].tolist(), visible=True
                ),
                text=grouped["count"].astype(int),
                # textposition="top center",
                name=severity,
            ),
            row=row,
            col=1,
        )
        fig.update_yaxes(title="IV days (mean ± SD)", row=row, col=1)
    fig.update_xaxes(title=f"{diff_col} binned", row=1, col=1)

    title = f"bFEV1_2016_19_IVdays_2019-21_binned_{prctile}th_prctile"

    fig.update_layout(
        xaxis_title=diff_col,
        title=title,
        height=800,
    )
    fig.write_image(dh.get_path_to_main() + f"PlotsCFR/Viz diff vs IV days/{title}.pdf")

50th percentile of absolute difference: 0.457945218826417
60th percentile of absolute difference: 2.781535403334735
70th percentile of absolute difference: 5.978852663315695
80th percentile of absolute difference: 12.69587666933772
90th percentile of absolute difference: 25.269813601145305
